# Kaggle Fine-Tuning Results — Visualization

Visualizes the artifacts produced by a Kaggle training run for **Task 1 (Risk Clause Recognition)** — binary `Yes`/`No` clause identification.

The notebook reads from a single results directory (default: `kaggle_output/`) and is organized in three parts:

1. **Setup & loaders** — reusable functions, no plotting.
2. **Training & evaluation data** — what the model was trained / evaluated on.
3. **Results** — training dynamics and validation performance.

Everything is modular: point `RESULTS_DIR` at any run directory with the same layout (e.g. `kaggle_output_3.2B_Llama_smoketest/`) and re-run.

## 1. Setup & reusable loaders

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

# ---- Configuration: point this at any run directory with the standard layout ----
RESULTS_DIR = Path("kaggle_output")

# Consistent, simple plot styling reused across the notebook.
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

POS, NEG = "#2a9d8f", "#e76f51"  # Yes / No color pair, reused everywhere

assert RESULTS_DIR.exists(), f"Results dir not found: {RESULTS_DIR.resolve()}"
print("Reading from:", RESULTS_DIR.resolve())

In [ ]:
def load_jsonl(path):
    """Load a .jsonl file into a DataFrame."""
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return pd.DataFrame(rows)


def load_json(path):
    """Load a single JSON file into a dict."""
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def find_trainer_state(results_dir):
    """Return the trainer_state.json from the highest-numbered checkpoint."""
    checkpoints = sorted(
        (results_dir / "results").glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if not checkpoints:
        return None
    return checkpoints[-1] / "trainer_state.json"


def bar_labels(ax, fmt="{:.0f}"):
    """Annotate each bar in an axis with its height."""
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2, h),
                    ha="center", va="bottom", fontsize=9)

In [ ]:
# Load every artifact once; downstream cells just read these.
train_df = load_jsonl(RESULTS_DIR / "cuad" / "train" / "cuad_train.jsonl")
val_df = load_jsonl(RESULTS_DIR / "cuad" / "validation" / "cuad_validation.jsonl")

train_metrics = load_json(RESULTS_DIR / "train_metrics.json")
eval_metrics = load_json(RESULTS_DIR / "eval_metrics.json")

_ts_path = find_trainer_state(RESULTS_DIR)
trainer_state = load_json(_ts_path) if _ts_path else None

print(f"train: {len(train_df)} examples | val: {len(val_df)} examples")
train_df.head(2)

## 2. Training & evaluation data

What the model saw. The data is instruction-formatted: each example asks whether a clause is present and expects a `Yes`/`No` answer.

In [ ]:
def split_sizes(splits):
    """Bar chart of example counts per split."""
    fig, ax = plt.subplots()
    ax.bar(splits.keys(), [len(df) for df in splits.values()], color=[POS, NEG])
    ax.set_title("Examples per split")
    ax.set_ylabel("# examples")
    bar_labels(ax)
    plt.tight_layout()
    plt.show()


split_sizes({"train": train_df, "validation": val_df})

In [ ]:
def label_balance(splits, label_col="output"):
    """Grouped bar chart of Yes/No label balance across splits."""
    counts = pd.DataFrame({
        name: df[label_col].value_counts() for name, df in splits.items()
    }).fillna(0)
    counts = counts.reindex(["Yes", "No"]).dropna(how="all")

    ax = counts.plot.bar(color={"train": POS, "validation": NEG})
    ax.set_title("Label balance (Yes / No)")
    ax.set_xlabel("label")
    ax.set_ylabel("# examples")
    ax.tick_params(axis="x", rotation=0)
    bar_labels(ax)
    plt.tight_layout()
    plt.show()

    # Print the positive-class rate, the key imbalance signal for T1.
    for name, df in splits.items():
        rate = (df[label_col] == "Yes").mean()
        print(f"{name:>11}: {rate:.1%} positive (Yes)")


label_balance({"train": train_df, "validation": val_df})

In [ ]:
def top_categories(df, n=15, title="Top clause categories (train)"):
    """Horizontal bar chart of the most frequent clause categories."""
    counts = df["category"].value_counts().head(n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, max(4, 0.35 * len(counts))))
    ax.barh(counts.index, counts.values, color=POS)
    ax.set_title(title)
    ax.set_xlabel("# examples")
    plt.tight_layout()
    plt.show()
    print(f"{df['category'].nunique()} distinct categories total")


if "category" in train_df.columns:
    top_categories(train_df)

In [ ]:
def input_length_hist(splits, text_col="input", bins=40):
    """Histogram of input clause lengths (in words) per split."""
    fig, ax = plt.subplots()
    for (name, df), color in zip(splits.items(), [POS, NEG]):
        lengths = df[text_col].str.split().str.len()
        ax.hist(lengths, bins=bins, alpha=0.6, label=name, color=color)
        print(f"{name:>11}: median {lengths.median():.0f} words, "
              f"95th pct {lengths.quantile(0.95):.0f}, max {lengths.max():.0f}")
    ax.set_title("Input clause length distribution")
    ax.set_xlabel("words per input")
    ax.set_ylabel("# examples")
    ax.legend()
    plt.tight_layout()
    plt.show()


input_length_hist({"train": train_df, "validation": val_df})

## 3. Results

Training dynamics first, then validation performance.

In [ ]:
def training_curves(trainer_state):
    """Plot loss and token accuracy over training steps."""
    log = pd.DataFrame(trainer_state["log_history"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(log["step"], log["loss"], marker="o", color=NEG)
    axes[0].set_title("Training loss")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("loss")

    if "mean_token_accuracy" in log:
        axes[1].plot(log["step"], log["mean_token_accuracy"], marker="o", color=POS)
        axes[1].set_title("Mean token accuracy")
        axes[1].set_xlabel("step")
        axes[1].set_ylabel("accuracy")

    plt.tight_layout()
    plt.show()


if trainer_state:
    training_curves(trainer_state)
    print(f"final training loss: {train_metrics['final_training_loss']:.4f}")
    print(f"steps: {train_metrics['global_step']} | "
          f"runtime: {train_metrics['train_runtime_seconds'] / 60:.1f} min")

In [ ]:
def run_summary(train_metrics, eval_metrics):
    """Compact table of the headline numbers for the run."""
    hp = train_metrics.get("hyperparameters", {})
    rows = {
        "base model": train_metrics.get("model_name"),
        "validation examples": eval_metrics.get("n_validation_examples"),
        "accuracy": f"{eval_metrics['accuracy']:.1%}",
        "epochs": hp.get("num_train_epochs"),
        "learning rate": hp.get("learning_rate"),
        "LoRA r / alpha": f"{hp.get('lora_r')} / {hp.get('lora_alpha')}",
        "max length": hp.get("max_length"),
    }
    return pd.DataFrame(rows.items(), columns=["metric", "value"]).set_index("metric")


run_summary(train_metrics, eval_metrics)

In [ ]:
def per_class_metrics(eval_metrics, classes=("Yes", "No")):
    """Grouped bar chart of precision / recall / F1 per class."""
    report = eval_metrics["classification_report"]
    metrics = ["precision", "recall", "f1-score"]
    data = pd.DataFrame(
        {m: [report[c][m] for c in classes] for m in metrics},
        index=list(classes),
    )
    ax = data.plot.bar(ylim=(0, 1.05))
    ax.set_title("Per-class performance (validation)")
    ax.set_xlabel("class")
    ax.set_ylabel("score")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="lower right")
    bar_labels(ax, fmt="{:.2f}")
    plt.tight_layout()
    plt.show()


per_class_metrics(eval_metrics)

In [ ]:
def confusion_matrix(eval_metrics):
    """Heatmap of the confusion matrix with raw counts."""
    cm = eval_metrics["confusion_matrix"]
    labels = cm["labels"]
    matrix = pd.DataFrame(cm["matrix"], index=labels, columns=labels)

    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title("Confusion matrix")
    ax.grid(False)
    thresh = matrix.values.max() / 2
    for i in range(len(labels)):
        for j in range(len(labels)):
            v = matrix.iloc[i, j]
            ax.text(j, i, str(v), ha="center", va="center",
                    color="white" if v > thresh else "black", fontsize=12)
    plt.tight_layout()
    plt.show()


confusion_matrix(eval_metrics)

### Takeaways

- The fine-tuned adapter reaches strong overall accuracy on the held-out validation set.
- `No` (clause absent) is the majority class; the `Yes` recall vs. precision gap reflects the class imbalance noted in the project's data decisions.
- To compare another run, change `RESULTS_DIR` in the setup cell and re-run all cells.